# Multimodal, RAG, dan Fine-Tuning

Notebook ini merangkum `03-multimodal-dan-rag.md`: representasi multimodal sederhana, embedding similarity, chunking, RAG dasar, hybrid retrieval, dan catatan advanced untuk vision fine-tuning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## API LLM Opsional

Notebook ini tetap bisa jalan tanpa API key. Jika ingin memanggil contoh LLM atau VLM, isi `GEMINI_API_KEY` melalui Kaggle Secrets atau environment variable. API key bisa dibuat dari Google AI Studio; free tier memiliki kuota dan dapat berubah. Di Kaggle, pastikan internet notebook aktif.

In [ ]:
import os
import io
import json
import base64
import urllib.request
import urllib.error

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(name, "")

GEMINI_API_KEY = get_secret("GEMINI_API_KEY") or "PASTE_GEMINI_API_KEY_HERE"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")

def call_gemini_payload(payload, model=GEMINI_MODEL):
    if not GEMINI_API_KEY or GEMINI_API_KEY.startswith("PASTE_"):
        return "[SKIP] Isi GEMINI_API_KEY di Kaggle Secrets atau environment variable untuk memanggil LLM."

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json", "x-goog-api-key": GEMINI_API_KEY},
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8")[:500]
        return f"[ERROR] Gemini API {error.code}: {detail}"

    return data["candidates"][0]["content"]["parts"][0]["text"]

def call_gemini_text(prompt):
    payload = {"contents": [{"parts": [{"text": prompt}]}]}
    return call_gemini_payload(payload)

def call_gemini_vision(prompt, image):
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    encoded_image = base64.b64encode(buffer.getvalue()).decode("utf-8")
    payload = {
        "contents": [{
            "parts": [
                {"text": prompt},
                {"inline_data": {"mime_type": "image/png", "data": encoded_image}},
            ]
        }]
    }
    return call_gemini_payload(payload)

## 1. Multimodal Toy Example

Model multimodal asli memakai encoder visual. Di sini kita memakai fitur warna sederhana agar konsep encoder tetap mudah terlihat.

In [ ]:
def make_image(color, label):
    image = Image.new("RGB", (96, 96), color=color)
    draw = ImageDraw.Draw(image)
    draw.rectangle((24, 24, 72, 72), outline=(255, 255, 255), width=3)
    return image

images = {
    "produk merah": make_image((220, 40, 40), "red"),
    "produk hijau": make_image((40, 170, 80), "green"),
    "produk biru": make_image((40, 90, 220), "blue"),
}

fig, axes = plt.subplots(1, 3, figsize=(7, 2.5))
for axis, (title, image) in zip(axes, images.items()):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def simple_image_embedding(image):
    pixels = np.asarray(image).astype(float) / 255.0
    return pixels.mean(axis=(0, 1))

image_embeddings = {name: simple_image_embedding(image) for name, image in images.items()}
for name, embedding in image_embeddings.items():
    print(name, np.round(embedding, 3))

## 2. Contoh Pemanggilan VLM

Cell ini mengirim gambar sederhana ke model multimodal jika `GEMINI_API_KEY` tersedia.

In [ ]:
vision_prompt = "Deskripsikan gambar ini dalam satu kalimat bahasa Indonesia."
print(call_gemini_vision(vision_prompt, images["produk hijau"]))

## 3. Text Retrieval dengan Cosine Similarity

Untuk praktikum ringan, TF-IDF dipakai sebagai embedding teks lokal. Di sistem RAG production, bagian ini biasanya diganti embedding model khusus.

In [ ]:
documents = [
    "Produk merah cocok untuk kampanye diskon akhir tahun.",
    "Produk hijau dipakai untuk materi ramah lingkungan.",
    "Produk biru menjadi pilihan utama untuk paket enterprise.",
    "Panduan retur berlaku selama tujuh hari setelah barang diterima.",
]
query = "produk untuk kampanye ramah lingkungan"

vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)
query_vector = vectorizer.transform([query])
scores = cosine_similarity(query_vector, doc_vectors).ravel()

for index in scores.argsort()[::-1]:
    print(round(scores[index], 3), documents[index])

## 4. Chunking dan RAG Dasar

RAG mengambil potongan dokumen yang relevan, lalu menyusun prompt agar jawaban model berbasis konteks.

In [ ]:
long_document = """
Produk merah dipakai untuk kampanye diskon dan promosi retail.
Produk hijau dipakai untuk kampanye ramah lingkungan dan edukasi sustainability.
Produk biru dipakai untuk klien enterprise yang memerlukan integrasi prioritas.
Retur produk hanya berlaku jika bukti pembelian masih tersedia.
""".strip()

chunks = [line.strip() for line in long_document.split("\n") if line.strip()]
chunk_vectors = vectorizer.fit_transform(chunks)
query_vector = vectorizer.transform([query])
chunk_scores = cosine_similarity(query_vector, chunk_vectors).ravel()
best_chunk = chunks[int(chunk_scores.argmax())]

augmented_prompt = f"""Jawab hanya berdasarkan konteks berikut.

Konteks:
{best_chunk}

Pertanyaan:
{query}
"""

print(augmented_prompt)

## 5. Jawaban LLM Berbasis RAG

Cell ini mengirim augmented prompt ke LLM jika `GEMINI_API_KEY` tersedia.

In [ ]:
print(call_gemini_text(augmented_prompt))

## 6. Hybrid Retrieval Sederhana

Hybrid RAG menggabungkan sinyal semantic search dan keyword search. Contoh ini sengaja sederhana agar mekanismenya terlihat.

In [ ]:
def keyword_score(query, text):
    query_words = set(query.lower().split())
    text_words = set(text.lower().replace(".", "").split())
    return len(query_words & text_words) / max(1, len(query_words))

semantic_scores = chunk_scores / max(chunk_scores.max(), 1e-9)
keyword_scores = np.array([keyword_score(query, chunk) for chunk in chunks])
hybrid_scores = 0.7 * semantic_scores + 0.3 * keyword_scores

for index in hybrid_scores.argsort()[::-1]:
    print(round(hybrid_scores[index], 3), chunks[index])

## 7. Advanced: Vision Fine-Tuning dengan Unsloth

Untuk praktik lanjutan, notebook vision fine-tuning dapat memakai Unsloth Qwen3.5 Vision:

- Colab: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb
- Cocok untuk eksplorasi mandiri karena butuh GPU, instalasi banyak dependency, dan waktu training.

Alur besarnya tetap sama: siapkan dataset gambar-teks, load VLM, pasang LoRA/adapter, train, uji inference, lalu simpan adapter.